# Bài 5.7 · CLIP: tìm ảnh bằng câu chữ

**Câu hỏi của buổi này:** một model chưa từng thấy nhãn lớp nào, chỉ học ghép ảnh với chú thích, thì biết gì về ảnh? Ta dùng **CLIP** trên 3 925 ảnh thật của **Imagenette** để tìm ảnh bằng câu chữ, rồi mở ra xem loss contrastive làm gì.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu và embedding | bảng số ảnh, số chiều |
| 2. Tìm ảnh bằng câu chữ | năm câu hỏi, mỗi câu năm ảnh khớp nhất |
| 3. Ma trận ảnh–chú thích | cosine của 8 ảnh với 8 câu; đường chéo sáng |
| 4. Temperature | softmax nhọn hay bẹt theo nhiệt độ |
| 5. Contrastive loss | tính tay loss trên một batch, khớp model |
| 6. Cặp dương và cặp âm | hai biến đổi của cùng ảnh so với hai ảnh khác nhau |
| 7. Zero-shot 10 lớp | accuracy, ảnh đoán sai |
| 8. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 3 phút. Bài giảng: [ai-canuc80k · bài 5.7](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/5.7).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, tarfile, urllib.request
from pathlib import Path
import torch, torchvision, transformers
import torch.nn.functional as F
from torchvision import transforms as T
from transformers import CLIPModel, CLIPProcessor
transformers.logging.set_verbosity_error()

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_IMG = 3925
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu và embedding

CLIP có hai encoder: ảnh thành một vector 512 chiều, câu chữ thành một vector 512 chiều, chuẩn hóa độ dài 1. Ảnh và câu mô tả đúng ảnh ấy được học để có cosine cao.

In [ ]:
root = Path('/tmp/imagenette')
if not (root / 'imagenette2-320').exists():
    root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve('https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz', root / 'data.tgz')
    with tarfile.open(root / 'data.tgz') as f: f.extractall(root)
ds = torchvision.datasets.ImageFolder(root / 'imagenette2-320' / 'val')
CLS = {'n01440764': 'tench', 'n02102040': 'English springer', 'n02979186': 'cassette player', 'n03000684': 'chain saw', 'n03028079': 'church',
       'n03394916': 'French horn', 'n03417042': 'garbage truck', 'n03425413': 'gas pump', 'n03445777': 'golf ball', 'n03888257': 'parachute'}
VI = ['cá tench', 'chó springer', 'máy cassette', 'cưa máy', 'nhà thờ', 'kèn French horn', 'xe rác', 'cây xăng', 'bóng golf', 'dù lượn']
names = [CLS[w] for w in ds.classes]
idx = np.random.default_rng(0).permutation(len(ds))[:N_IMG]
y = np.array(ds.targets)[idx]

with quiet():
    clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
    proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
feat = lambda out: out if torch.is_tensor(out) else out.pooler_output          # transformers 5 trả object
def img_emb(imgs):
    with torch.inference_mode():
        x = proc(images=imgs, return_tensors='pt')['pixel_values'].to(device)
        return F.normalize(feat(clip.get_image_features(pixel_values=x)).float(), dim=1)
def txt_emb(texts):
    with torch.inference_mode():
        t = proc(text=texts, return_tensors='pt', padding=True).to(device)
        return F.normalize(feat(clip.get_text_features(**t)).float(), dim=1)

t0 = time.time()
E = torch.cat([img_emb([ds[i][0] for i in idx[s:s + 128]]) for s in range(0, len(idx), 128)])
table(pd.DataFrame([('số ảnh', len(idx)), ('số lớp', len(names)), ('embedding ảnh', tuple(E.shape)), ('giây để mã hóa', f'{time.time() - t0:.0f}')], columns=['', 'giá trị']), 'Imagenette (validation) qua encoder ảnh của CLIP')
check('mỗi embedding có độ dài 1', torch.allclose(E.norm(dim=1), torch.ones(len(E), device=device), atol=1e-4))

## 2. Tìm ảnh bằng câu chữ

Mã hóa một câu tiếng Anh, tính cosine với 3 925 ảnh, lấy 5 ảnh cao nhất. Không có nhãn nào được dùng: các câu hỏi cố ý mô tả **chi tiết** hơn tên lớp.

In [ ]:
queries = ['a dog lying on the grass', 'a church at night', 'a person swinging a golf club', 'a big orange truck on the street', 'a man holding a large fish']
Q = txt_emb(queries)
S = (Q @ E.T).cpu()
fig, axes = plt.subplots(len(queries), 6, figsize=(14, 2.5 * len(queries)), gridspec_kw={'width_ratios': [1.6, 1, 1, 1, 1, 1]})
for r, q in enumerate(queries):
    axes[r, 0].text(0.02, 0.5, f'"{q}"', fontsize=11, va='center', wrap=True); axes[r, 0].axis('off')
    for c, j in enumerate(S[r].topk(5).indices, start=1):
        axes[r, c].imshow(ds[idx[j]][0]); axes[r, c].axis('off'); axes[r, c].set_title(f'{S[r, j]:.3f}', fontsize=9)
fig.suptitle('Năm ảnh có cosine cao nhất với mỗi câu (số trên ảnh là cosine)', y=1.0)
plt.tight_layout(); plt.show()
note('Cosine giữa ảnh và chữ của CLIP thường chỉ quanh 0.2–0.35; điều quan trọng là thứ tự, không phải giá trị tuyệt đối.')

## 3. Ma trận ảnh–chú thích

Lấy 8 ảnh của 8 lớp và 8 câu "a photo of a {lớp}". Ô $(i, j)$ là cosine giữa ảnh $i$ và câu $j$. Khi huấn luyện, CLIP được đẩy để **đường chéo** (cặp đúng) cao hơn mọi ô khác cùng hàng, cùng cột.

In [ ]:
pick = [int(np.flatnonzero(y == k)[0]) for k in range(8)]
caps = [f'a photo of a {names[k]}' for k in range(8)]
I8, T8 = E[pick], txt_emb(caps)
M = (I8 @ T8.T).cpu().numpy()
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for k, (ax, i) in enumerate(zip(axes, pick)):
    ax.imshow(ds[idx[i]][0]); ax.axis('off'); ax.set_title(f'ảnh {k}', fontsize=9)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(M, cmap='Blues'); plt.colorbar(im, ax=ax, fraction=.046)
for i in range(8):
    for j in range(8): ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=9, color='white' if M[i, j] > M.mean() + M.std() else '#333')
ax.set_xticks(range(8), [f'"a photo of a {names[k]}"' for k in range(8)], rotation=40, ha='right', fontsize=9); ax.set_yticks(range(8), [f'ảnh {k}: {VI[k]}' for k in range(8)], fontsize=9)
ax.set_title('cosine(ảnh, câu)'); ax.grid(False)
plt.tight_layout(); plt.show()
check('mỗi ảnh có cosine cao nhất với đúng câu của nó', (M.argmax(1) == np.arange(8)).all())

## 4. Temperature

Logits $= \text{cosine} / \tau$. CLIP học $1/\tau \approx 100$. Cùng một hàng cosine, softmax với $\tau$ lớn thì bẹt (model không dám chọn), $\tau$ nhỏ thì nhọn.

In [ ]:
scale = clip.logit_scale.exp().item()
row = torch.tensor(M[0])
fig, ax = plt.subplots(figsize=(10, 3.4))
xs = np.arange(8); w = .26
for k, (s, c) in enumerate([(1, C['gray']), (10, C['sand']), (scale, C['blue'])]):
    p = (row * s).softmax(0).numpy(); ax.bar(xs + (k - 1) * w, p, w, color=c, label=f'1/τ = {s:.0f}')
ax.set_xticks(xs, [names[k] for k in range(8)], rotation=20, fontsize=9); ax.set(ylabel='xác suất', title=f'Ảnh 0 ({VI[0]}): softmax trên 8 câu với ba nhiệt độ'); ax.legend()
plt.tight_layout(); plt.show()
ent = lambda p: -(p * p.clamp_min(1e-12).log()).sum().item()
tab = pd.DataFrame([(f'{s:.0f}', (row * s).softmax(0).max().item(), ent((row * s).softmax(0))) for s in [1, 10, scale]], columns=['1/τ', 'xác suất lớn nhất', 'entropy'])
table(tab, f'CLIP học được 1/τ = {scale:.1f}', fmt={'xác suất lớn nhất': '{:.3f}', 'entropy': '{:.3f}'})
note(f'Với 1/τ = 1, cosine chênh nhau vài phần trăm nên softmax gần đều (entropy {tab.iloc[0, 2]:.2f}, tối đa ln 8 = 2.08). Với 1/τ = {scale:.0f}, câu đúng chiếm {tab.iloc[2, 1]:.0%}.')

## 5. Contrastive loss

Với một batch $N$ cặp (ảnh, câu), logits $= \frac{1}{\tau} I T^\top$ ($N \times N$). Loss là cross-entropy theo hàng (mỗi ảnh chọn câu của nó) cộng theo cột (mỗi câu chọn ảnh của nó), chia đôi. Tính tay rồi so với loss mà model trả về.

In [ ]:
imgs = [ds[idx[i]][0] for i in pick]
with torch.inference_mode():
    inp = proc(text=caps, images=imgs, return_tensors='pt', padding=True).to(device)
    out = clip(**inp, return_loss=True)
logits = scale * (I8 @ T8.T)
target = torch.arange(8, device=device)
l_img, l_txt = F.cross_entropy(logits, target), F.cross_entropy(logits.T, target)
manual = (l_img + l_txt) / 2
table(pd.DataFrame([('ảnh → câu (theo hàng)', l_img.item()), ('câu → ảnh (theo cột)', l_txt.item()), ('trung bình (tự tính)', manual.item()), ('model trả về', out.loss.item()),
                    ('nếu chọn ngẫu nhiên: ln 8', np.log(8))], columns=['phần', 'loss']), 'Contrastive loss trên batch 8 cặp', fmt={'loss': '{:.4f}'})
check(f'tự tính khớp model (chênh {abs(manual.item() - out.loss.item()):.1e})', abs(manual.item() - out.loss.item()) < 1e-3)
shuf = (F.cross_entropy(logits, target.roll(1)) + F.cross_entropy(logits.T, target.roll(1))).item() / 2
note(f'Ghép sai mọi cặp (dời câu đi một chỗ) thì loss lên <b>{shuf:.2f}</b>: loss nhỏ khi và chỉ khi đường chéo thắng.')

## 6. Cặp dương và cặp âm

Self-supervised (như SimCLR) không có chú thích: lấy **hai biến đổi ngẫu nhiên của cùng một ảnh** làm cặp dương, hai ảnh khác nhau làm cặp âm. Encoder tốt phải cho cặp dương cosine cao hơn cặp âm. Kiểm trên encoder ảnh của CLIP với 200 ảnh.

In [ ]:
aug = T.Compose([T.RandomResizedCrop(224, scale=(0.3, 1.0)), T.RandomHorizontalFlip(), T.ColorJitter(0.4, 0.4, 0.4, 0.1)])
torch.manual_seed(0)
sub = idx[:200]
v1 = img_emb([aug(ds[i][0]) for i in sub]); v2 = img_emb([aug(ds[i][0]) for i in sub])
sim = (v1 @ v2.T).cpu().numpy()
pos = np.diag(sim); same = np.array(ds.targets)[sub]
neg_same = sim[(same[:, None] == same[None]) & ~np.eye(200, dtype=bool)]
neg_diff = sim[same[:, None] != same[None]]
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={'width_ratios': [1, 1.3]})
i0 = sub[3]; torch.manual_seed(1)
axes[0].imshow(np.hstack([np.asarray(ds[i0][0].resize((160, 160)))] + [np.asarray(aug(ds[i0][0]).resize((160, 160))) for _ in range(3)])); axes[0].axis('off')
axes[0].set_title('một ảnh gốc và ba biến đổi ngẫu nhiên của nó')
bins = np.linspace(0.3, 1, 50)
for v, c, n in [(neg_diff, C['gray'], 'hai ảnh khác lớp'), (neg_same, C['sand'], 'hai ảnh cùng lớp'), (pos, C['blue'], 'hai biến đổi của cùng ảnh')]:
    axes[1].hist(v, bins=bins, density=True, alpha=.6, color=c, label=f'{n} (trung bình {v.mean():.2f})')
axes[1].set(xlabel='cosine giữa hai embedding', ylabel='mật độ', title='Cặp dương nằm hẳn bên phải'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()
top1 = (sim.argmax(1) == np.arange(200)).mean()
check(f'cặp dương trung bình ({pos.mean():.2f}) cao hơn cặp âm ({neg_diff.mean():.2f})', pos.mean() > neg_diff.mean())
note(f'Với mỗi ảnh, trong 200 ứng viên, biến đổi của chính nó đứng đầu ở <b>{top1:.0%}</b> trường hợp. Hai ảnh cùng lớp cũng gần nhau hơn hai ảnh khác lớp, dù encoder không biết nhãn lớp.')

## 7. Zero-shot 10 lớp

In [ ]:
prompts = [f'a photo of a {n}.' for n in names]
P = (100 * E @ txt_emb(prompts).T).softmax(1).cpu()
pred = P.argmax(1).numpy()
acc = (pred == y).mean()
per = pd.DataFrame({'lớp': VI, 'accuracy': [(pred[y == k] == k).mean() for k in range(10)]}).sort_values('accuracy')
table(per.head(3), f'Zero-shot {acc:.3f} trên {len(y)} ảnh; ba lớp khó nhất', fmt={'accuracy': '{:.3f}'})
wrong = np.flatnonzero(pred != y)[:6]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(2.4 * len(wrong), 2.8))
    for ax, i in zip(np.atleast_1d(axes), wrong):
        ax.imshow(ds[idx[i]][0]); ax.axis('off'); ax.set_title(f'thật: {VI[y[i]]}\nđoán: {VI[pred[i]]}', fontsize=9, color=C['clay'])
    plt.tight_layout(); plt.show()
check('zero-shot vượt 0.9', acc > 0.9)

## 8. Tóm tắt

In [ ]:
summary = [('số ảnh tìm kiếm', f'{len(idx):,}'), ('1/τ CLIP học được', f'{scale:.1f}'),
           ('contrastive loss batch 8: đúng cặp / dời cặp', f'{manual.item():.3f} / {shuf:.3f}'),
           ('cosine cặp dương / cặp âm khác lớp', f'{pos.mean():.2f} / {neg_diff.mean():.2f}'),
           ('biến đổi của chính nó đứng đầu', f'{top1:.0%}'), ('zero-shot 10 lớp', f'{acc:.3f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** viết câu hỏi của chính lớp học ở mục 2 (thử cả câu tiếng Việt và xem CLIP gốc yếu thế nào); giảm `scale=(0.3, 1.0)` xuống `(0.08, 1.0)` ở mục 6 để cặp dương khó hơn.